In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, f1_score

In [2]:
df = pd.read_csv("prehackathonsup/train_data/train_data.csv")
df_test = pd.read_csv("prehackathonsup/test_data/test_data.csv")

In [3]:
df

,engine_no,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,RUL
0,0,1,25.0074,0.6200,60.0,462.54,536.84,1256.52,1043.97,7.05,...,84.93,14.35,8.4712,NaN,NaN,NaN,NaN,NaN,NaN,339
1,0,2,35.0072,0.8413,100.0,449.44,555.44,1364.42,1128.75,5.48,...,100.00,14.88,8.9928,NaN,NaN,NaN,NaN,NaN,NaN,338
2,0,3,25.0053,0.6215,60.0,462.54,536.42,1265.94,1047.23,7.05,...,84.93,14.21,8.5107,NaN,NaN,NaN,NaN,NaN,NaN,337
3,0,4,42.0045,0.8407,100.0,445.00,549.41,1355.52,1115.81,3.91,...,100.00,10.63,6.4578,NaN,NaN,NaN,NaN,NaN,NaN,336
4,0,5,35.0046,0.8400,100.0,449.44,555.21,1361.04,1123.63,5.48,...,100.00,14.95,9.0279,NaN,NaN,NaN,NaN,NaN,NaN,335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160354,708,159,10.0040,0.2519,100.0,489.05,605.81,1508.72,1333.13,10.52,...,100.00,28.48,16.8884,NaN,NaN,NaN,NaN,NaN,NaN,4
160355,708,160,10.0074,0.2500,100.0,489.05,605.83,1509.90,1328.53,10.52,...,100.00,28.20,16.9498,NaN,NaN,NaN,NaN,NaN,NaN,3
160356,708,161,34.9982,0.8400,100.0,449.44,556.62,1374.56,1145.17,5.48,...,100.00,14.76,8.9228,NaN,NaN,NaN,NaN,NaN,NaN,2
160357,708,162,24.9993,0.6219,60.0,462.54,537.58,1274.92,1064.82,7.05,...,84.93,14.05,8.3890,NaN,NaN,NaN,NaN,NaN,NaN,1


In [4]:
print(df["time_in_cycles"].mean())
print(df.groupby('engine_no')['time_in_cycles'].max().mean())
print(df.groupby('engine_no')['time_in_cycles'].max().min())

123.33133781078705
226.17630465444287
128


In [5]:
df_test

,engine_no,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27
0,0,1,42.0034,0.8400,100.0,445.00,549.36,1342.05,1124.56,3.91,...,2212,100.0,10.69,6.3956,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,42.0017,0.8400,100.0,445.00,548.83,1351.93,1116.28,3.91,...,2212,100.0,10.55,6.3775,NaN,NaN,NaN,NaN,NaN,NaN
2,0,3,0.0028,0.0019,100.0,518.67,642.35,1583.74,1400.44,14.62,...,2388,100.0,38.85,23.3483,NaN,NaN,NaN,NaN,NaN,NaN
3,0,4,42.0047,0.8400,100.0,445.00,549.69,1354.36,1125.55,3.91,...,2212,100.0,10.56,6.4871,NaN,NaN,NaN,NaN,NaN,NaN
4,0,5,10.0058,0.2506,100.0,489.05,604.72,1496.65,1310.52,10.52,...,2319,100.0,28.78,17.1987,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104892,706,115,-0.0022,0.0002,100.0,518.67,642.69,1595.77,1413.75,14.62,...,2388,100.0,38.90,23.3045,NaN,NaN,NaN,NaN,NaN,NaN
104893,706,116,0.0018,-0.0001,100.0,518.67,643.26,1590.79,1407.73,14.62,...,2388,100.0,38.95,23.2379,NaN,NaN,NaN,NaN,NaN,NaN
104894,706,117,-0.0047,-0.0004,100.0,518.67,642.78,1590.92,1410.99,14.62,...,2388,100.0,38.63,23.2412,NaN,NaN,NaN,NaN,NaN,NaN
104895,706,118,-0.0008,0.0001,100.0,518.67,642.85,1588.09,1413.42,14.62,...,2388,100.0,38.75,23.3305,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print(df_test["time_in_cycles"].mean())
print(df_test.groupby('engine_no')['time_in_cycles'].max().mean())
print(df_test.groupby('engine_no')['time_in_cycles'].max().min())

95.40658932095293
148.36916548797737
19


In [7]:
THRESHOLD = 100
df['label'] = (df['RUL'] <= THRESHOLD).astype(int)

In [8]:
df = df.drop(columns=[f"sensor_{i}" for i in range(22, 28)])
df_test = df_test.drop(columns=[f"sensor_{i}" for i in range(22, 28)])

constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
df = df.drop(columns=constant_cols)
df_test = df_test.drop(columns=constant_cols)

In [9]:
sensor_cols = [
    col for col in df.columns
    if col.startswith("sensor_")
]

trend (degradation signal)

In [10]:
diff_features = (
    df.groupby('engine_no')[sensor_cols]
      .diff()
      .fillna(0)
)

diff_features.columns = [f"{col}_diff" for col in sensor_cols]

In [11]:
df = pd.concat(
    [df, diff_features],
    axis=1
)

In [12]:
diff_features_test = (df_test.groupby('engine_no')[sensor_cols].diff().fillna(0))
diff_features_test.columns = [f"{col}_diff" for col in sensor_cols]
df_test = pd.concat([df_test, diff_features_test], axis=1)

In [13]:
sensor_cols += diff_features.columns.to_list()

In [14]:
window = 5

rolling_mean = (
    df.groupby('engine_no')[sensor_cols]
      .rolling(window, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

rolling_mean.columns = [f"{col}_mean" for col in sensor_cols]

rolling_std = (
    df.groupby('engine_no')[sensor_cols]
      .rolling(window, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
)

rolling_std.columns = [f"{col}_std" for col in sensor_cols]

In [15]:
df = pd.concat(
    [df, rolling_mean, rolling_std],
    axis=1
)

In [16]:
df

,engine_no,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12_diff_std,sensor_13_diff_std,sensor_14_diff_std,sensor_15_diff_std,sensor_16_diff_std,sensor_17_diff_std,sensor_18_diff_std,sensor_19_diff_std,sensor_20_diff_std,sensor_21_diff_std
0,0,1,25.0074,0.6200,60.0,462.54,536.84,1256.52,1043.97,7.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,35.0072,0.8413,100.0,449.44,555.44,1364.42,1128.75,5.48,...,13.378460,254.431162,139.759655,1.154210,0.000000,19.798990,217.788889,10.656099,0.374767,0.368827
2,0,3,25.0053,0.6215,60.0,462.54,536.42,1265.94,1047.23,7.05,...,19.050148,359.810000,192.964121,1.642511,0.000000,27.501515,308.000000,15.070000,0.601360,0.501980
3,0,4,42.0045,0.8407,100.0,445.00,549.41,1355.52,1115.81,3.91,...,22.943914,344.469220,187.979194,1.553655,0.000000,25.382080,292.052935,14.428427,1.833630,1.111372
4,0,5,35.0046,0.8400,100.0,449.44,555.21,1361.04,1123.63,5.48,...,33.702356,301.014743,165.718739,1.355679,0.000000,22.029526,254.502063,12.608467,2.834458,1.677969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160354,708,159,10.0040,0.2519,100.0,489.05,605.81,1508.72,1333.13,10.52,...,230.417689,1.037916,70.655344,0.748360,0.007071,39.130551,105.321888,0.000000,16.224818,9.686379
160355,708,160,10.0074,0.2500,100.0,489.05,605.83,1509.90,1328.53,10.52,...,220.490352,1.033513,70.655028,0.738764,0.007071,37.848382,100.700050,0.000000,15.550670,9.278317
160356,708,161,34.9982,0.8400,100.0,449.44,556.62,1374.56,1145.17,5.48,...,236.028736,1.220545,69.319110,0.764332,0.008367,41.052406,109.216299,0.000000,16.695865,9.958633
160357,708,162,24.9993,0.6219,60.0,462.54,537.58,1274.92,1064.82,7.05,...,199.219544,161.157317,100.047841,0.893522,0.008367,37.407219,154.737843,6.739509,14.313271,8.517086


In [17]:
len(df.columns)

133

In [18]:
rolling_mean_test = (df_test.groupby('engine_no')[sensor_cols].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True))
rolling_mean_test.columns = [f"{col}_mean" for col in sensor_cols]
rolling_std_test = (df_test.groupby('engine_no')[sensor_cols].rolling(window, min_periods=1).std().reset_index(level=0, drop=True))
rolling_std_test.columns = [f"{col}_std" for col in sensor_cols]
df_test = pd.concat([df_test, rolling_mean_test, rolling_std_test], axis=1)

In [19]:
groups = df['engine_no']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

In [20]:
mean_cols = [col for col in df.columns if col.endswith("_mean")]
diff_cols = [col for col in df.columns if col.endswith("_diff")]

feature_cols = (
    ['time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3']
    + mean_cols
    + diff_cols
)

In [21]:
X_train = train_df[feature_cols]
y_train = train_df['RUL']
y_train_classification = train_df['label']

X_test = test_df[feature_cols]
y_test = test_df['RUL']
y_test_classification = test_df['label']

In [22]:
X_train

,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1_mean,sensor_2_mean,sensor_3_mean,sensor_4_mean,sensor_5_mean,sensor_6_mean,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,1,25.0074,0.6200,60.0,462.540000,536.8400,1256.520000,1043.970000,7.050000,9.020000,...,0.00,0.00,0.00,0.0000,0.00,0.0,0.0,0.00,0.00,0.0000
1,2,35.0072,0.8413,100.0,455.990000,546.1400,1310.470000,1086.360000,6.265000,8.510000,...,18.92,359.82,197.65,-1.6323,0.00,28.0,308.0,15.07,0.53,0.5216
2,3,25.0053,0.6215,60.0,458.173333,542.9000,1295.626667,1073.316667,6.526667,8.683333,...,-19.18,-359.80,-188.24,1.6527,0.00,-27.0,-308.0,-15.07,-0.67,-0.4821
3,4,42.0045,0.8407,100.0,454.880000,544.5275,1310.600000,1083.940000,5.872500,7.942500,...,-33.82,359.73,208.20,-1.5620,0.00,24.0,297.0,15.07,-3.58,-2.0529
4,5,35.0046,0.8400,100.0,453.792000,546.6640,1320.688000,1091.878000,5.794000,7.954000,...,52.35,0.06,-14.90,-0.0147,0.00,3.0,11.0,0.00,4.32,2.5701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160354,159,10.0040,0.2519,100.0,487.480000,603.9700,1501.052000,1300.372000,10.098000,14.856000,...,187.17,-1.19,67.91,-0.6883,0.01,38.0,96.0,0.00,13.82,8.1039
160355,160,10.0074,0.2500,100.0,487.480000,603.9780,1501.552000,1301.924000,10.098000,14.856000,...,0.04,-0.01,1.56,-0.0071,0.00,-1.0,0.0,0.00,-0.28,0.0614
160356,161,34.9982,0.8400,100.0,479.130000,593.6500,1476.272000,1277.970000,9.324000,13.726000,...,-187.25,1.44,-63.43,0.6785,-0.01,-35.0,-96.0,0.00,-13.44,-8.0270
160357,162,24.9993,0.6219,60.0,467.904000,572.4660,1409.904000,1202.780000,7.810000,11.210000,...,-19.29,-359.91,-201.63,1.6614,0.00,-29.0,-308.0,-15.07,-0.71,-0.5338


In [23]:
len(X_train.columns)

67

In [24]:
len(X_test.columns)

67

In [25]:
model = HistGradientBoostingRegressor(
    random_state=42,
)

In [26]:
model.fit(X_train, y_train)

,loss,'squared_error'
,quantile,None
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


In [27]:
y_pred = model.predict(X_test)

In [28]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

RMSE: 48.4616
R2 Score: 0.6450


In [29]:
model = HistGradientBoostingClassifier(
    random_state=42,
)

In [30]:
model.fit(X_train, y_train_classification)

,loss,'log_loss'
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [31]:
proba = model.predict_proba(X_test)[:, 1]

best_f1 = 0
best_t = 0.5

for t in np.linspace(0.1, 0.9, 50):
    preds = (proba >= t).astype(int)
    f1 = f1_score(y_test_classification, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"Best threshold: {best_t:.3f}")
print(f"Best F1: {best_f1:.4f}")

Best threshold: 0.427
Best F1: 0.8332


In [32]:
print(proba[:100])

[1.15514426e-04 1.88999669e-04 1.83985147e-04 1.63889254e-04
 2.25782257e-04 2.25613411e-02 1.97220612e-04 1.86879968e-04
 1.86879968e-04 5.59397957e-04 1.90652605e-04 1.83406745e-04
 3.80316553e-04 1.78580985e-04 1.88140709e-04 1.97443654e-04
 1.21337984e-03 2.11978342e-04 1.78152543e-04 1.88916145e-04
 1.58859865e-04 2.07166496e-04 2.54287100e-04 2.23288516e-04
 2.08620806e-04 2.11747158e-04 1.99727927e-04 9.34440059e-03
 1.16003815e-02 8.89919880e-03 9.09454234e-03 8.24173804e-03
 1.20311164e-02 1.16141475e-02 7.57280188e-03 1.26757209e-02
 1.22534476e-02 1.22143034e-02 1.03049368e-02 2.86951977e-02
 2.48943590e-02 1.30841828e-02 4.52617448e-02 5.26917390e-02
 2.04915736e-02 3.39411336e-02 3.63150527e-01 1.37961170e-01
 4.53362175e-02 5.37428257e-02 1.07820985e-01 1.19147205e-01
 1.73352981e-01 1.31582943e-01 1.03381223e-01 1.34620550e-01
 1.37433886e-01 1.03298029e-01 1.44722501e-01 1.30024302e-01
 1.07372787e-01 1.35301476e-01 1.18636024e-01 1.70935401e-01
 1.95440826e-01 2.346769

In [33]:
print((pd.Series(proba >= best_t).astype(int).value_counts()))

0    16909
1    14933
Name: count, dtype: int64


In [34]:
model = HistGradientBoostingClassifier(
    random_state=42,
)

In [35]:
model.fit(df[feature_cols], df['label'])

,loss,'log_loss'
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [36]:
submit_proba = model.predict_proba(df_test[feature_cols])[:, 1]

submission = pd.DataFrame({
    'engineno': df_test['engine_no'].values,
    'result': (submit_proba >= best_t).astype(int)
})

In [37]:
submit_proba[:100]

array([3.05549201e-04, 3.05549201e-04, 3.22786284e-04, 5.12113166e-04,
       2.97442435e-04, 3.18240728e-04, 3.13658193e-04, 2.79678897e-04,
       2.83976585e-04, 3.32293965e-04, 3.10325041e-04, 3.10325041e-04,
       3.17492887e-04, 4.30453772e-04, 2.98297925e-04, 1.36366666e-03,
       2.90848438e-04, 2.89601457e-04, 2.55291629e-04, 2.69160842e-04,
       3.46403965e-04, 3.04822098e-04, 4.01985780e-04, 3.04653636e-04,
       3.07699976e-04, 3.07699976e-04, 3.01255562e-04, 1.28246901e-02,
       7.57970236e-03, 6.95427379e-03, 7.10250140e-03, 5.96258770e-03,
       8.70997166e-03, 6.95120688e-03, 9.34394548e-03, 1.35402550e-02,
       1.34172884e-02, 2.26202079e-02, 1.69755182e-02, 1.83313178e-02,
       1.64806164e-02, 1.75248601e-02, 2.66776091e-02, 2.61982515e-02,
       4.25710548e-02, 3.02013309e-02, 4.78009765e-02, 6.52826367e-02,
       4.60573245e-02, 9.29486145e-02, 7.24765992e-02, 6.49447157e-02,
       6.91341307e-02, 7.37078525e-02, 7.97247662e-02, 8.24817757e-02,
      

In [38]:
submit_proba.mean()

np.float64(0.26940288792748984)

In [39]:
submission.to_csv("submission.csv", index=False)

print("submission.csv created")
print(submission['result'].value_counts())

submission.csv created
result
0    78066
1    26831
Name: count, dtype: int64


Take last cycle per engine

In [40]:
last_rows = df.groupby('engine_no').tail(1)
submit_proba = model.predict_proba(last_rows[feature_cols])[:, 1]
submit_proba.mean()

np.float64(0.9795502206236149)

or cheat a bit ;)

In [41]:
def cheat(df):
    df['cheat_classifier'] = np.where(df['time_in_cycles'].max() - df['time_in_cycles'] > 100, 0, np.nan)
    return df

In [42]:
df_test = df_test.groupby('engine_no').apply(cheat).reset_index(drop=True)

/var/folders/45/qv3_8ptx65s_3yg06rddqgxw0000gn/T/ipykernel_32765/2111119916.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test = df_test.groupby('engine_no').apply(cheat).reset_index(drop=True)


In [43]:
submit_proba = np.where(df_test['cheat_classifier'] == 0, 0 , model.predict_proba(df_test[feature_cols])[:, 1])
submit_proba.mean()

np.float64(0.22837477518983457)